# Autonomous Blockchain Investigator — GPU Training & Ablation on Elliptic++
Run this notebook on a **Google Colab T4 GPU** (Runtime → Change runtime type → T4 GPU) or **Vertex AI Workbench**.
It authenticates to your GCP project (`capstone-bitcoin-forensics`), pulls the real **Elliptic++ CSVs** from `gs://capstone-bitcoin-forensics-abi-vault`, runs the **Static vs. Temporal Bit-CHetG** ablation, and uploads the trained `.pt` checkpoint straight back to GCS so your live Cloud Run microservice automatically loads it.

In [ ]:
# 1. Authenticate Colab to your GCP account
from google.colab import auth
auth.authenticate_user()
!gcloud config set project capstone-bitcoin-forensics

In [ ]:
# 2. Clone your GitHub repo & install dependencies
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'  # <-- Replace with your repo URL
!git clone $REPO_URL abi_repo
%cd abi_repo
!pip install -q -e ./contracts
!pip install -q google-cloud-storage networkx pandas pydantic

In [ ]:
# 3. Train Static vs. Temporal Bit-CHetG on real Elliptic++ (from GCS) + AMLSim typologies on GPU
!python training/train_elliptic_pp_gpu.py --use-gcs --bucket capstone-bitcoin-forensics-abi-vault --epochs 25 --max-elliptic-txs 40000

In [ ]:
# 4. Hot-reload your live Cloud Run GNN service with the freshly trained GCS checkpoint!
MOD2_URL = 'https://YOUR_MOD2_CLOUD_RUN_URL'  # <-- Copy from GitHub Actions Deploy Summary
!curl -X POST $MOD2_URL/reload-checkpoint